# Phase 5 — Legal Statute Identification with Mamba + a bidirectional head

**Idea.** The Phase-1 Mamba model is a *causal* (left-to-right) sequence model: the state at token *t* has only seen tokens ≤ *t*. Exactly like the GQLA notebook, we keep it as a
pre-trained legal-text encoder and put **bidirectional** sequence models on top of its representations:

```
document → 512-token chunks (same window as pre-training)
        → Mamba hidden states (frozen, optionally top-k blocks unfrozen)
        → [token-level BiGRU/BiLSTM]  → one vector per chunk
        → chunk-level BiGRU / BiLSTM / small BERT-style encoder
        → attention pooling → sigmoid per statute  (multi-label)
```

Two modes share everything after the Mamba backbone (identical to the GQLA Phase-5 notebook):
* **`cached`** – Mamba is run once, each chunk stored as `[mean | max | last-token]` of its states; only the chunk-level bidirectional
  encoder is trained. Seconds per epoch → use this for the ablations. (For Mamba the *last-token* vector is the end of the recurrent state, i.e. a summary of the whole chunk.)
* **`token`** – literal token-level BiRNN over Mamba states, then the chunk-level encoder. Slower, richer; supports `unfreeze_top_k`.

**Macro-F1 is the primary metric**, so the notebook keeps the rare-statute-aware losses (`bce_pw`, `asl`), the force-at-least-one-statute rule, and
global / per-statute threshold tuning with the *honest* split-half check.

### What is faster here, and why
| Piece | What it does | Where it applies |
|---|---|---|
| **`mamba_ssm` fused selective-scan kernel** | one CUDA kernel for discretisation + scan + `D·x` skip + `SiLU(z)` gate (`softplus(dt+bias)` is fused too); no `(B, L, d_inner, d_state)` tensors ever materialised | Mamba forward in feature extraction, token-mode training, inference |
| **Pure-PyTorch fallback scan** (used automatically if the kernel is missing) | the Phase-1 chunked scan, now run in 128-step segments so memory stays bounded and bigger batches fit | same places |
| **fp16 / bf16 autocast** | projections and conv in half precision; the scan state stays fp32 inside the kernel; residual stream stays fp32 | everywhere on GPU (auto-checked against the fp32 reference) |
| **Token-budget batching** | length-sorted chunks, batch sized by a token budget, one preallocated output buffer | feature extraction |
| **FlashAttention via `F.scaled_dot_product_attention`** | the BERT-style chunk encoder (`chunk_rnn="transformer"`) uses SDPA, which dispatches to the FlashAttention kernel under fp16/bf16 on Ampere-or-newer GPUs when no padding mask is needed (otherwise, e.g. on a T4, to the memory-efficient kernel) | chunk-level encoder only |

> **Honest note on flash attention.** Mamba itself contains *no attention*, so there is nothing to flash-ify inside the backbone. FlashAttention can only help the optional
> Transformer chunk encoder (sequences of ≤ `max_chunks` chunk vectors), where it is a small saving. The big speed-ups come from the Mamba scan kernel, autocast and batching.

**Run order:** run all cells top to bottom. Needs the Phase-1 Mamba checkpoint `mamba_large.pt` (written by the Phase-1 Mamba notebook) and the shared tokenizer `final_bpe_en.json`.
Edit `MAMBA_CFG` if you trained with different hyper-parameters. The *self-test cell* (after the model heads) verifies the fast path against the Phase-1 pure-PyTorch maths before any features are extracted, and switches the kernel / half precision off by itself if they disagree.


---
## Phase-5 experiment plan (brief, sec. 11.2): 6 experiments for the Mamba architecture
| # | Experiment | Technique | What is trained | Where it lives |
|---|---|---|---|---|
| E1 | `e1_lorma_plus` | **LoRMA+** (multiplicative low-rank, compulsory) | B, A on `in_proj`/`out_proj` of the top blocks + BiRNN head | `LoRMALinear` |
| E2 | `e2_adalora` | **AdaLoRA** (SVD-style update, importance-based rank pruning, orthogonality regulariser) | P, Λ, Q + head | `AdaLoRALinear`, `AdaLoRAAllocator` |
| E3 | `e3_qlora` | **QLoRA** (frozen 4-bit NF4 Mamba + LoRA) | LoRA A, B + head | `NF4Linear` |
| E4 | `e4_adapter` | **Bottleneck adapters** (Houlsby/Pfeiffer style) | one adapter per block + head | `BottleneckAdapter` |
| E5 | `e5_partial_ft` | **Partial / layer-wise fine-tuning** | top-k Mamba blocks + head | `MambaBackbone.unfreeze_blocks` |
| E6 | `e6_kd_student` | **Knowledge distillation** | smaller student (8 blocks, small head) learns from the E1–E5 ensemble | `kd=` in `run_experiment` |

Extras (commented in the RUN cell): LoRMAπ, prompt tuning, full fine-tuning, a student without distillation. Every run records trainable parameters, deployed model size, train time, dev-inference time and peak GPU memory, so the "model size / memory / inference cost" question of the brief can be answered from the summary table.
The brief also asks for the *other* Phase-1 architectures; this notebook only covers the Mamba one.

In [ ]:
# ============================================================
# OPTIONAL (GPU runtime): fused Mamba CUDA kernels
#   - mamba-ssm      -> fused selective scan (the big one)
#   - causal-conv1d  -> fused causal conv; supports kernel width 2-4 ONLY. Your Phase-1 model
#                       has d_conv=16, so this notebook cannot use it (the conv runs through cuDNN).
# Pre-built wheels exist only for some torch/CUDA/python combinations; otherwise pip compiles from
# source (can take 10-30 min on Colab). The notebook works without it (slower pure-PyTorch scan).
# Restart the runtime after installing, then run everything from the top.
# ============================================================
# !pip install -q ninja packaging
!pip install -q mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user


In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

In [ ]:
# ============================================================
# normalize_text - copied VERBATIM from the Phase-1 Mamba notebook's cleaning cell.
# Must be identical for training and inference (Phase-5 documents get the same normalisation).
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text

In [ ]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step

In [ ]:
device = get_device()
print("Using device: ", device)

Using device:  cuda


In [ ]:
# ============================================================
# 0. SETUP
# Run this notebook in the SAME Colab runtime as your Phase-1 notebook (or just make sure the Drive
# paths below exist). BPETokenizer and normalize_text are defined in the cells above (copied
# from Phase 1: they MUST be identical for training and inference); the Mamba classes are in the next cell.
# ============================================================
import os, json, math, time, random, copy, pickle, shutil, hashlib
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

# ---- Hugging Face access ---------------------------------------------------------------------------------------
# Exploration-Lab/CS-781-Capstone is a GATED dataset: without a token that has access you get "401 ... restricted".
# 1) open https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone while logged in and accept/request access;
# 2) create a read token (HF > Settings > Access Tokens) and either add it as a Colab secret named HF_TOKEN
#    (switch "Notebook access" on; secrets are only readable when the notebook is run from the Colab UI),
#    or paste it below. Never share a notebook that has a token typed into it.
HF_TOKEN = os.environ.get("HF_TOKEN")            # e.g. HF_TOKEN = "hf_xxx"
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("WARNING: no HF_TOKEN found - the data download below will fail with a 401 unless you are already logged in.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")

# ---- paths (edit if yours differ) ------------------------------------------
cache_dir      = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
ckpt_path      = os.path.join(cache_dir, "mamba_checkpoints", "mamba_large.pt")   # Phase-1 Mamba (latest resumable ckpt; has model_state_dict)
tok_path       = os.path.join(cache_dir, "final_bpe_en.json")
p5_dir         = os.path.join(cache_dir, "phase5_mamba"); os.makedirs(p5_dir, exist_ok=True)   # features + results of THIS notebook
tok_share_dir  = os.path.join(cache_dir, "phase5")        # tokenised Phase-5 docs from the GQLA notebook: reused if present (same tokenizer + cleaning)

# ---- Mamba hyper-parameters: MUST equal what you trained with (Phase-1 Mamba notebook, model cell) -------
MAMBA_CFG = dict(d_model=512, n_layers=20, d_state=32, d_conv=16, expand=3, dropout=0.1)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID = tokenizer.stoi["<pad>"]
print("tokenizer vocab:", len(tokenizer.itos))

Mounted at /content/drive
device: cuda | NVIDIA A100-SXM4-80GB
tokenizer vocab: 32768


In [ ]:
# ============================================================
# MAMBA (Gu & Dao, 2023) - the Phase-1 architecture, with an optional fused CUDA scan.
#
# Parameter / module names are IDENTICAL to the Phase-1 notebook's MambaLM, so the Phase-1
# checkpoint loads with strict=True. Differences from Phase 1 (maths unchanged):
#   * forward() can run the fused `mamba_ssm` selective-scan kernel: softplus(dt + bias), the
#     scan, the D*x skip and the SiLU(z) gate are one CUDA launch.
#   * the pure-PyTorch fallback scan now runs in 128-step segments (carrying the SSM state), so
#     memory is bounded by the segment instead of the whole 512-token chunk.
#   * no generate()/step(): Phase 5 only needs hidden states.
# ============================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# ---- optional fused kernels ---------------------------------------------------------------
HAS_SSM_KERNEL, HAS_CONV_KERNEL, _KERNEL_ERR = False, False, None
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_SSM_KERNEL = True
except Exception as _e:                                   # not installed / no matching build
    selective_scan_fn, _KERNEL_ERR = None, repr(_e)
try:
    from causal_conv1d import causal_conv1d_fn
    HAS_CONV_KERNEL = True
except Exception:
    causal_conv1d_fn = None

# runtime switches (the self-test cell may turn the scan kernel off if it disagrees with the reference)
FAST = {
    "scan_kernel": bool(HAS_SSM_KERNEL and torch.cuda.is_available()),
    "conv_kernel": bool(HAS_CONV_KERNEL and torch.cuda.is_available()),
}
print("mamba_ssm selective-scan kernel :", "available" if HAS_SSM_KERNEL else f"NOT available ({_KERNEL_ERR})")
print("causal-conv1d kernel            :", "available" if HAS_CONV_KERNEL else "not available (only usable for d_conv <= 4)")
print("scan kernel in use              :", FAST["scan_kernel"])

_MIN_LOG_DECAY = -8.0            # Phase-1 clamp on per-step log(a_t) = dt * A (pure-PyTorch path only)


# ============================================================
# Pure-PyTorch scan (Phase-1 maths):  h_t = exp(dt_t A) h_{t-1} + dt_t B_t x_t ;  y_t = C_t . h_t + D x_t
# ============================================================
def selective_scan_chunked(x, dt, A, Bm, Cm, D, chunk=8, h0=None):
    """Chunked parallel scan in fp32.  x, dt: (B, L, Din)  A: (Din, N)  Bm, Cm: (B, L, N)  D: (Din,)
    h0: optional initial state (B, Din, N).  Returns y (B, L, Din) and the final state (B, Din, N)."""
    Bsz, L, Din = x.shape
    N = A.shape[1]

    pad = (-L) % chunk                       # zero padding is a no-op for the state (dt = 0 -> a = 1, b = 0)
    if pad:
        x, dt = F.pad(x, (0, 0, 0, pad)), F.pad(dt, (0, 0, 0, pad))
        Bm, Cm = F.pad(Bm, (0, 0, 0, pad)), F.pad(Cm, (0, 0, 0, pad))
    Lp = L + pad
    nc = Lp // chunk

    dA = (dt.unsqueeze(-1) * A).clamp(min=_MIN_LOG_DECAY)        # (B, Lp, Din, N), <= 0
    dBx = (dt * x).unsqueeze(-1) * Bm.unsqueeze(2)               # (B, Lp, Din, N)
    dA = dA.view(Bsz, nc, chunk, Din, N)
    dBx = dBx.view(Bsz, nc, chunk, Din, N)

    La = torch.cumsum(dA, dim=2)                                  # within-chunk cumulative log-decay
    local = torch.exp(La) * torch.cumsum(dBx * torch.exp(-La), dim=2)   # chunk-local states (zero initial state)

    chunk_decay = torch.exp(La[:, :, -1])                         # (B, nc, Din, N)
    chunk_end = local[:, :, -1]                                   # (B, nc, Din, N)

    carry = x.new_zeros(Bsz, Din, N) if h0 is None else h0
    prev = []
    for k in range(nc):                                           # only L/chunk iterations
        prev.append(carry)
        carry = chunk_decay[:, k] * carry + chunk_end[:, k]
    h_last = carry
    prev = torch.stack(prev, dim=1)                               # state entering each chunk

    h = local + torch.exp(La) * prev.unsqueeze(2)                 # (B, nc, chunk, Din, N)
    y = (h * Cm.view(Bsz, nc, chunk, 1, N)).sum(-1).reshape(Bsz, Lp, Din)[:, :L]
    y = y + x[:, :L] * D
    return y, h_last


def selective_scan_segmented(x, dt, A, Bm, Cm, D, chunk=8, seg=128):
    """Same result as selective_scan_chunked, but the (B, seg, Din, N) intermediates only exist for
    `seg` steps at a time (the SSM state is carried between segments). `seg` must be a multiple of `chunk`."""
    L = x.size(1)
    if L <= seg:
        return selective_scan_chunked(x, dt, A, Bm, Cm, D, chunk)
    ys, h = [], None
    for s in range(0, L, seg):
        y, h = selective_scan_chunked(x[:, s:s + seg], dt[:, s:s + seg], A,
                                      Bm[:, s:s + seg], Cm[:, s:s + seg], D, chunk, h0=h)
        ys.append(y)
    return torch.cat(ys, dim=1), h


# ============================================================
# Norm
# ============================================================
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


def rms_normalize(x, eps=1e-5):
    """Parameter-free RMS norm (used to read out intermediate blocks, where no learned norm exists)."""
    xf = x.float()
    return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(x.dtype)


# ============================================================
# Mamba mixer (selective SSM part of the block, Fig. 3 of the paper)
# ============================================================
class MambaMixer(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank=None,
                 dt_min=1e-3, dt_max=1e-1, dt_init_floor=1e-4, scan_chunk=8):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = expand * d_model
        self.dt_rank = math.ceil(d_model / 16) if dt_rank is None else dt_rank
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)          # -> (x, z)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                groups=self.d_inner, padding=d_conv - 1, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)  # -> (dt, B, C)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        nn.init.uniform_(self.dt_proj.weight, -self.dt_rank ** -0.5, self.dt_rank ** -0.5)
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
                       ).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))                               # softplus^-1
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.A_log._no_weight_decay = True
        self.D._no_weight_decay = True

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def _conv_silu(self, x):
        """x: (B, Din, L) -> SiLU(causal depthwise conv(x)), (B, Din, L)."""
        L = x.size(-1)
        if FAST["conv_kernel"] and self.d_conv <= 4 and x.is_cuda:
            return causal_conv1d_fn(x.contiguous(), self.conv1d.weight.squeeze(1), self.conv1d.bias,
                                    activation="silu")
        return F.silu(self.conv1d(x)[..., :L])

    def forward(self, u):
        """u: (B, L, d_model) -> (B, L, d_model)"""
        x, z = self.in_proj(u).chunk(2, dim=-1)                                   # (B, L, Din) each
        x = self._conv_silu(x.transpose(1, 2))                                    # (B, Din, L)
        dt, Bm, Cm = torch.split(self.x_proj(x.transpose(1, 2)),
                                 [self.dt_rank, self.d_state, self.d_state], dim=-1)
        A = -torch.exp(self.A_log.float())                                        # (Din, N)

        if FAST["scan_kernel"] and u.is_cuda:
            # fused kernel: softplus(dt + bias), scan, D*x and the SiLU(z) gate in one launch.
            # (It does not apply Phase-1's -8 clamp on dt*A; that only changes how much of an already
            #  fully-decayed state survives, <= ~3e-4 relative. The self-test cell measures the net effect.)
            dt = F.linear(dt, self.dt_proj.weight)                                # bias is added inside the kernel
            cd = x.dtype                                                          # u, dt, B, C, z must share a dtype
            y = selective_scan_fn(
                x.contiguous(),
                dt.to(cd).transpose(1, 2).contiguous(),
                A,
                Bm.to(cd).transpose(1, 2).contiguous(),
                Cm.to(cd).transpose(1, 2).contiguous(),
                self.D.float(),
                z=z.to(cd).transpose(1, 2).contiguous(),
                delta_bias=self.dt_proj.bias.float(),
                delta_softplus=True,
            )
            y = y.transpose(1, 2)                                                 # (B, L, Din), already gated
        else:
            dt = F.softplus(self.dt_proj(dt))                                     # (B, L, Din)
            with torch.autocast(device_type=u.device.type, enabled=False):        # scan always fp32
                y, _ = selective_scan_segmented(
                    x.transpose(1, 2).float(), dt.float(), A, Bm.float(), Cm.float(),
                    self.D.float(), chunk=self.scan_chunk)
            y = y.to(z.dtype) * F.silu(z)                                         # gate branch
        return self.out_proj(y)


# ============================================================
# Block (pre-norm residual) and the Phase-1 language model (weights are loaded, never re-initialised)
# ============================================================
class MambaBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1, **mixer_kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mixer = MambaMixer(d_model, **mixer_kwargs)
        self.dropout = nn.Dropout(dropout)
        self.adapter = None            # optional bottleneck adapter (Phase-5 PEFT); None adds no parameters / state_dict keys

    def forward(self, x):
        y = self.mixer(self.norm(x))
        if self.adapter is not None:
            y = self.adapter(y)
        return x + self.dropout(y)


class MambaLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_state=16, d_conv=4, expand=2,
                 dropout=0.1, pad_token_id=0, scan_chunk=8):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, dropout=dropout, d_state=d_state, d_conv=d_conv,
                       expand=expand, scan_chunk=scan_chunk)
            for _ in range(n_layers)
        ])
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)   # only so the checkpoint loads strictly

    def forward(self, input_ids):                                    # full LM logits (not used in Phase 5)
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.norm_final(x))

mamba_ssm selective-scan kernel : NOT available (ModuleNotFoundError("No module named 'mamba_ssm'"))
causal-conv1d kernel            : not available (only usable for d_conv <= 4)
scan kernel in use              : False


In [ ]:
def load_pretrained_mamba():
    m = MambaLM(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **MAMBA_CFG)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"])          # strict: fails loudly on a config mismatch
    m.lm_head = nn.Identity()                          # next-token head is not needed for Phase 5 (frees ~17M params)
    return m.eval()


MAMBA = load_pretrained_mamba()
CKPT_SIG = hashlib.md5(f"{os.path.basename(ckpt_path)}_{os.path.getsize(ckpt_path)}".encode()).hexdigest()[:8]
print(f"Mamba loaded: {sum(p.numel() for p in MAMBA.parameters())/1e6:.1f}M params "
      f"(without LM head) | {len(MAMBA.layers)} blocks | ckpt signature {CKPT_SIG}")

Mamba loaded: 69.5M params (without LM head) | 20 blocks | ckpt signature aaa2f3a8


In [ ]:
# ============================================================
# 1. LOAD PHASE-5 (LSI) DATA
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-5 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset")
    hits = [f for f in files if f.endswith("/" + basename) and ("lsi" in f.lower() or "phase_5" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 5, got {hits}. "
                           f"Set the path by hand. All files: {files[:40]}")
    return hits[0]


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset")
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                       # {id: {text, labels}}  (Phase-1 style)
        raw = [{**v, "id": k} for k, v in raw.items()]
    recs = []
    for r in raw:                                   # [{id, text, labels}]  (Phase-5 spec)
        recs.append(dict(id=r["id"], text=r["text"], labels=[int(l) for l in r.get("labels", [])]))
    return recs


train_recs, dev_recs, test_recs = read_split("train.json"), read_split("dev.json"), read_split("test.json")
print(len(train_recs), len(dev_recs), len(test_recs), "train/dev/test documents")

# ---- label space: compact index <-> statute id --------------------------------
all_labels = sorted({l for r in train_recs + dev_recs for l in r["labels"]})
lab2idx = {l: i for i, l in enumerate(all_labels)}
idx2lab = {i: l for l, i in lab2idx.items()}
L = len(all_labels)


def multi_hot(recs):
    y = np.zeros((len(recs), L), dtype=np.float32)
    for i, r in enumerate(recs):
        for l in r["labels"]:
            y[i, lab2idx[l]] = 1.0
    return y


y_train, y_dev = multi_hot(train_recs), multi_hot(dev_recs)
freq = y_train.sum(0)
print(f"{L} statutes | labels/doc: train {y_train.sum(1).mean():.2f}")
print(f"train docs per statute: min {freq.min():.0f}  median {np.median(freq):.0f}  max {freq.max():.0f}")
print(f"statutes with <=5 train docs: {(freq <= 5).sum()}   in dev but never in train: "
      f"{((y_dev.sum(0) > 0) & (freq == 0)).sum()}")

# ---- tokenise once (full documents), cache to Drive -----------------------------
def encode_docs(recs, tag, bs=2000, workers=min(4, os.cpu_count() or 1)):
    path = os.path.join(p5_dir, f"tok_{tag}.pkl")
    for cand in (path, os.path.join(tok_share_dir, f"tok_{tag}.pkl")):     # reuse the GQLA notebook's tokenisation if it exists
        if os.path.exists(cand):
            return pickle.load(open(cand, "rb"))
    out = []
    for i in range(0, len(recs), bs):
        texts = [normalize_text(r["text"]) for r in recs[i:i + bs]]        # same normalisation as Phase 1
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(texts, add_special_tokens=True, num_workers=workers, chunk_size=32)]
    pickle.dump(out, open(path, "wb"))
    return out


enc_train, enc_dev, enc_test = (encode_docs(train_recs, "train"), encode_docs(dev_recs, "dev"),
                                encode_docs(test_recs, "test"))
lens = np.array([len(x) for x in enc_train])
print("train tokens/doc  median %d  p90 %d  p99 %d  max %d" % tuple(np.percentile(lens, [50, 90, 99, 100]).astype(int)))
print("-> 512-token chunks/doc  median %.1f  p90 %.1f" % tuple(np.percentile(np.ceil(lens / 512), [50, 90])))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

42750 10181 13019 train/dev/test documents
100 statutes | labels/doc: train 3.70
train docs per statute: min 272  median 679  max 11730
statutes with <=5 train docs: 0   in dev but never in train: 0
train tokens/doc  median 636  p90 4150  p99 12306  max 140079
-> 512-token chunks/doc  median 2.0  p90 9.0


In [ ]:
# ============================================================
# Pure-numpy utilities: chunking, metrics, thresholds
# (no torch here on purpose, so they are easy to unit-test)
# ============================================================
import numpy as np


# ---------------- long-document chunking ----------------
def chunk_ids(ids, chunk_len=512, stride=512, min_tail=16):
    """Split one tokenised document into windows of <= chunk_len tokens.
    stride == chunk_len  -> the same non-overlapping windows used in Phase-1 pre-training.
    stride <  chunk_len  -> overlapping windows."""
    if len(ids) <= chunk_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + chunk_len])
        if s + chunk_len >= len(ids):
            break
        s += stride
    # a tiny trailing fragment carries almost no information
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="head_tail"):
    """Long-document representation strategy (one of the ablations)."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(f"unknown strategy {strategy!r}")


# ---------------- metrics ----------------
def _counts(y, pred):
    tp = (y * pred).sum(0)
    fp = ((1 - y) * pred).sum(0)
    fn = (y * (1 - pred)).sum(0)
    return tp, fp, fn


def f1_per_class(y, pred):
    tp, fp, fn = _counts(y, pred)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0), y.sum(0)


def compute_metrics(y, pred):
    """macro_f1      : mean F1 over statutes that occur in `y` (primary metric)
       macro_f1_all  : mean F1 over every column, absent statutes count as 0
       micro_f1, subset_acc : the two secondary metrics"""
    f1, sup = f1_per_class(y, pred)
    tp, fp, fn = _counts(y, pred)
    return dict(
        macro_f1=float(f1[sup > 0].mean()) if (sup > 0).any() else 0.0,
        macro_f1_all=float(f1.mean()),
        micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        subset_acc=float((y == pred).all(1).mean()),
    )


# ---------------- thresholding ----------------
THR_GRID = np.round(np.arange(0.05, 0.951, 0.025), 4)


def predict_labels(probs, thr, force_top1=True):
    """thr: scalar or per-class array [L]. Every document has >=1 statute, so if
    nothing clears the threshold we fall back to the arg-max statute."""
    pred = (probs >= thr).astype(np.float32)
    if force_top1:
        empty = np.where(pred.sum(1) == 0)[0]
        if len(empty):
            pred[empty, probs[empty].argmax(1)] = 1.0
    return pred


def tune_global_threshold(probs, y, grid=THR_GRID, force_top1=True):
    scores = [compute_metrics(y, predict_labels(probs, t, force_top1))["macro_f1"] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])


def tune_per_class_threshold(probs, y, global_thr, grid=THR_GRID, min_support=3, shrink=0.5):
    """Per-statute thresholds. Statutes with < min_support positives in the tuning
    set keep the global threshold (too little evidence to tune); the rest are
    shrunk towards it:  thr_c = shrink*best_c + (1-shrink)*global."""
    n, L = probs.shape
    thr = np.full(L, global_thr, dtype=np.float32)
    sup = y.sum(0)
    for c in range(L):
        if sup[c] < min_support:
            continue
        P = probs[:, c:c + 1] >= grid[None, :]
        yc = y[:, c:c + 1] > 0
        tp = (P & yc).sum(0)
        fp = (P & ~yc).sum(0)
        fn = (~P & yc).sum(0)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        best = grid[np.flatnonzero(f1 == f1.max())]           # ties -> closest to global
        best = best[np.argmin(np.abs(best - global_thr))]
        thr[c] = shrink * best + (1 - shrink) * global_thr
    return thr


def honest_threshold_eval(probs, y, min_support=2, shrink=0.5, seed=0):
    """Tuning thresholds on dev and scoring on the same dev is optimistic. This
    tunes on one random half of dev and scores on the other half (and swaps)."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(y))
    A, B = perm[: len(y) // 2], perm[len(y) // 2:]
    res = {"fixed_0.5": [], "global": [], "per_class": []}
    for tr, te in ((A, B), (B, A)):
        g, _ = tune_global_threshold(probs[tr], y[tr])
        pc = tune_per_class_threshold(probs[tr], y[tr], g, min_support=min_support, shrink=shrink)
        res["fixed_0.5"].append(compute_metrics(y[te], predict_labels(probs[te], 0.5))["macro_f1"])
        res["global"].append(compute_metrics(y[te], predict_labels(probs[te], g))["macro_f1"])
        res["per_class"].append(compute_metrics(y[te], predict_labels(probs[te], pc))["macro_f1"])
    return {k: float(np.mean(v)) for k, v in res.items()}


def rare_report(y_dev, pred, y_train, edges=((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9))):
    """Mean F1 on dev, bucketed by how many TRAIN documents carry the statute."""
    f1, sup = f1_per_class(y_dev, pred)
    tf = y_train.sum(0)
    rows = []
    for lo, hi in edges:
        m = (tf >= lo) & (tf <= hi) & (sup > 0)
        rows.append((f"{lo}-{hi if hi < 10**9 else 'inf'}", int(m.sum()),
                     float(f1[m].mean()) if m.any() else float("nan")))
    return rows

In [ ]:
# ============================================================
# 2. MODEL:  Mamba (causal)  ->  BiGRU/BiLSTM over tokens  ->  BiGRU/BiLSTM/BERT-style encoder over chunks
#
# Why a bidirectional head?  Mamba is a left-to-right recurrence: the state at token t has only seen
# tokens <= t, so early tokens know nothing about the rest of the chunk. A bidirectional layer on top of
# the Mamba states lets every position see its left AND right context. Legal documents are far longer
# than the 512 tokens Mamba was trained on, so we do it hierarchically:
#
#   doc -> chunks of <=512 tokens -> [frozen Mamba states] -> token-level BiRNN -> chunk vector
#       -> chunk-level BiRNN (or small bidirectional Transformer) -> attention pooling -> sigmoid per statute
# ============================================================
from contextlib import nullcontext


def _pack_run(rnn, x, lengths):
    """Run an RNN over right-padded x with packing so the backward direction starts at the real end."""
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
    out, _ = rnn(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
    return out


def _make_rnn(kind, d_in, hid, layers, dropout):
    cls = {"gru": nn.GRU, "lstm": nn.LSTM}[kind]
    return cls(d_in, hid, layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)


class MambaBackbone(nn.Module):
    """Wraps the pre-trained MambaLM and returns hidden states (no LM head).
       use_layer      : -1 = final block (after norm_final); k>=0 = output of block k (parameter-free RMS norm).
                        Intermediate blocks are often better features than the last, next-token-specialised one.
                        (Your model has 20 blocks, so e.g. use_layer=10 is the middle.)
       unfreeze_top_k : 0 = fully frozen; k>0 = fine-tune the top k USED blocks (+ final norm if use_layer=-1).
       PEFT methods (cell 'PEFT for Mamba') inject trainable modules into the used blocks and lower
       `first_grad_block`, the first block that must run with autograd. Blocks below it run under no_grad
       (no activation memory); blocks from it upwards are gradient-checkpointed.
       The Mamba stays in eval mode (no dropout) even when the wrapper is in train mode.
       Causality makes right-padding safe: real tokens never see the pad positions (scan and conv are causal)."""

    def __init__(self, mamba, use_layer=-1, unfreeze_top_k=0, grad_ckpt=True):
        super().__init__()
        self.mamba, self.use_layer, self.grad_ckpt = mamba, use_layer, grad_ckpt
        self.d_model = mamba.token_embedding.embedding_dim
        n = len(mamba.layers)
        self.n_used = n if use_layer == -1 else use_layer + 1          # blocks that contribute to the output
        for p in mamba.parameters():
            p.requires_grad = False
        self.first_grad_block = self.n_used                             # = nothing trainable inside the Mamba
        self.soft_prompt, self.train_embed = None, False
        if unfreeze_top_k > 0:
            self.unfreeze_blocks(max(0, self.n_used - unfreeze_top_k))

    def unfreeze_blocks(self, first):
        for blk in self.mamba.layers[first:self.n_used]:
            for p in blk.parameters():
                p.requires_grad = True
        if self.use_layer == -1:
            for p in self.mamba.norm_final.parameters():
                p.requires_grad = True
        self.first_grad_block = min(self.first_grad_block, first)

    def train(self, mode=True):
        super().train(mode)
        self.mamba.eval()
        return self

    def forward(self, ids):                                   # ids [M,T] -> [M,T,d]
        m = self.mamba
        lo = min(self.first_grad_block, self.n_used)
        with torch.set_grad_enabled(self.train_embed and torch.is_grad_enabled()):
            h = m.token_embedding(ids)
        P = 0
        if self.soft_prompt is not None:                      # prompt tuning: learned vectors prepended to every chunk
            P = self.soft_prompt.size(0)
            h = torch.cat([self.soft_prompt.unsqueeze(0).expand(h.size(0), -1, -1).to(h.dtype), h], dim=1)
        with torch.no_grad() if lo > 0 else nullcontext():             # frozen lower blocks: no graph, no memory
            for blk in m.layers[:lo]:
                h = blk(h)
        for blk in m.layers[lo:self.n_used]:                  # blocks that need gradients (empty when frozen)
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                h = grad_checkpoint(blk, h, use_reentrant=False)
            else:
                h = blk(h)
        h = m.norm_final(h) if self.use_layer == -1 else rms_normalize(h)
        return h[:, P:] if P else h


class TokenBiRNN(nn.Module):
    """Bidirectional RNN over the Mamba token states of ONE chunk -> a single chunk vector (mean+max pool)."""

    def __init__(self, d_in, hid=128, kind="gru", layers=1, dropout=0.1):
        super().__init__()
        self.ln, self.drop = nn.LayerNorm(d_in), nn.Dropout(dropout)
        self.rnn = _make_rnn(kind, d_in, hid, layers, dropout)
        self.out_dim = 4 * hid

    def forward(self, h, tok_mask):                           # [M,T,d], [M,T] bool
        lengths = tok_mask.sum(1)
        out = _pack_run(self.rnn, self.drop(self.ln(h)), lengths).float()     # pool in fp32 even under autocast
        m = tok_mask.unsqueeze(-1)
        mean = (out * m).sum(1) / lengths.unsqueeze(-1)
        mx = out.masked_fill(~m, -1e4).max(1).values
        return torch.cat([mean, mx], -1)


class SDPAEncoderLayer(nn.Module):
    """Pre-norm bidirectional Transformer layer whose attention is F.scaled_dot_product_attention.
       SDPA dispatches to the FlashAttention kernel when the inputs are fp16/bf16 on an Ampere-or-newer GPU and NO mask
       is needed (all documents in the batch have the same number of chunks); with padding, or on older GPUs such as a
       T4, it falls back to the memory-efficient kernel. It is mathematically the same as nn.TransformerEncoderLayer(norm_first=True)."""

    def __init__(self, d, heads, d_ff, dropout):
        super().__init__()
        assert d % heads == 0
        self.h, self.p = heads, dropout
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_mask):                           # x [B,N,D]; key_mask [B,N] True = real chunk
        B, N, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        mask = None if bool(key_mask.all()) else key_mask[:, None, None, :]   # bool mask: True = may attend
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0)
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class ChunkSeqEncoder(nn.Module):
    """Contextualises the sequence of chunk vectors.
       kind: 'gru' | 'lstm'  -> bidirectional RNN
             'transformer'   -> small BERT-style bidirectional encoder (learned chunk-position embeddings, SDPA/flash attention)
             'none'          -> no cross-chunk modelling (ablation baseline)"""

    def __init__(self, d_in, hid=256, kind="gru", layers=1, dropout=0.3, max_chunks=64, heads=4):
        super().__init__()
        self.kind, D = kind, 2 * hid
        self.inp = nn.Sequential(nn.LayerNorm(d_in), nn.Dropout(dropout), nn.Linear(d_in, D), nn.GELU())
        if kind in ("gru", "lstm"):
            self.rnn = _make_rnn(kind, D, hid, layers, dropout)
        elif kind == "transformer":
            self.pos = nn.Embedding(max_chunks, D)
            self.enc = nn.ModuleList([SDPAEncoderLayer(D, heads, 4 * D, dropout) for _ in range(layers)])
            self.norm_out = nn.LayerNorm(D)
        elif kind != "none":
            raise ValueError(kind)
        self.out_dim = D

    def forward(self, x, chunk_mask):                         # [B,N,d_in], [B,N] bool
        z = self.inp(x)
        if self.kind in ("gru", "lstm"):
            z = _pack_run(self.rnn, z, chunk_mask.sum(1))
        elif self.kind == "transformer":
            z = z + self.pos(torch.arange(z.size(1), device=z.device))
            for layer in self.enc:
                z = layer(z, chunk_mask)
            z = self.norm_out(z)
        return z


class DocHead(nn.Module):
    """Chunk sequence -> one logit per statute.
       'attn'       : one attention over chunks, shared linear classifier (few parameters, safest for rare statutes)
       'label_attn' : every statute has its own attention query and weight vector, i.e. each statute picks the chunks
                      that matter for it (more capacity; self.last_attn [B,L,N] shows which chunk fired for which statute)
       'meanmax'    : masked mean+max pooling"""

    def __init__(self, dim, n_labels, pool="attn", dropout=0.3):
        super().__init__()
        self.pool = pool
        self.drop = nn.Dropout(dropout)
        self.last_attn = None
        if pool == "attn":
            self.att = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
            self.fc = nn.Linear(dim, n_labels)
        elif pool == "label_attn":
            self.Q = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.Q)
            self.W = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.W)
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif pool == "meanmax":
            self.fc = nn.Linear(2 * dim, n_labels)
        else:
            raise ValueError(pool)

    def forward(self, z, mask):                               # [B,N,D], [B,N]
        if self.pool == "attn":
            s = self.att(z).squeeze(-1).masked_fill(~mask, -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            return self.fc(self.drop((a.unsqueeze(-1) * z).sum(1)))
        if self.pool == "label_attn":
            s = torch.einsum("bnd,ld->bln", z, self.Q).masked_fill(~mask[:, None, :], -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            ctx = torch.einsum("bln,bnd->bld", a, z)
            return (self.drop(ctx) * self.W).sum(-1) + self.b
        m = mask.unsqueeze(-1)
        mean = (z * m).sum(1) / m.sum(1)
        mx = z.masked_fill(~m, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], -1)))


class MambaBiRNNClassifier(nn.Module):
    """MODE 'token': token-level BiRNN on Mamba states (literal 'BiRNN on top of Mamba representations').
       The Mamba forward is re-run every step (it is frozen, so under no_grad), in micro-batches of chunks."""

    def __init__(self, mamba, n_labels, cfg):
        super().__init__()
        self.backbone = MambaBackbone(mamba, cfg.use_layer, cfg.unfreeze_top_k)
        # PEFT / fine-tuning method for this experiment (None = frozen backbone). See cell 'PEFT for Mamba'.
        self.peft = apply_peft(self.backbone, cfg.peft) if getattr(cfg, "peft", None) else None
        self.tok = TokenBiRNN(self.backbone.d_model, cfg.tok_hid, cfg.tok_rnn, 1, 0.1)
        self.chunk = ChunkSeqEncoder(self.tok.out_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers,
                                     cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)
        self.chunk_bs = cfg.mamba_chunk_bs

    def forward(self, ids, tok_mask, chunk_mask):             # [B,N,T], [B,N,T], [B,N]
        B, N, T = ids.shape
        flat_ids, flat_tm = ids[chunk_mask], tok_mask[chunk_mask]      # only real chunks: [M,T]
        vecs = [self.tok(self.backbone(flat_ids[i:i + self.chunk_bs]), flat_tm[i:i + self.chunk_bs])
                for i in range(0, flat_ids.size(0), self.chunk_bs)]
        vecs = torch.cat(vecs)
        x = vecs.new_zeros(B, N, vecs.size(-1))
        x[chunk_mask] = vecs
        return self.head(self.chunk(x, chunk_mask), chunk_mask)


class CachedFeatureClassifier(nn.Module):
    """MODE 'cached': Mamba frozen and run ONCE; every chunk is stored as [mean | max | last-token] of its Mamba states.
       Only the chunk-level bidirectional encoder + head are trained -> seconds per epoch, ideal for ablations.
       The 'last-token' vector is the one that has seen the whole chunk (causal), so it is a natural summary."""

    def __init__(self, in_dim, n_labels, cfg):
        super().__init__()
        self.chunk = ChunkSeqEncoder(in_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)

    def forward(self, feats, chunk_mask):
        return self.head(self.chunk(feats, chunk_mask), chunk_mask)

In [ ]:
# ============================================================
# PEFT FOR MAMBA  -  the adaptation techniques of the Phase-5 brief, applied to the Phase-1 Mamba
#
#   method        technique (reference)                              what is trained
#   ------------  -------------------------------------------------  ------------------------------------------
#   lorma_plus    LoRMA+  (Bihany, Patel, Modi, ACL-Findings 2025)    B, A of  (a/r*BA + I) W0   [multiplicative]
#   lorma_pi      LoRMA-pi (same paper, permutation inflation)        B, A of  I_pi(BA) W0
#   adalora       AdaLoRA (Zhang et al., 2023)                        P, Lambda, Q (SVD form) + rank pruning
#   qlora         QLoRA   (Dettmers et al., 2023)                     LoRA on top of a frozen 4-bit NF4 Mamba
#   adapter       Bottleneck adapters (Houlsby 2019 / Pfeiffer 2021)  down/up projections after each mixer
#   prompt        Prompt tuning (Lester et al., 2021)                 P soft tokens prepended to every chunk
#   partial       Partial / layer-wise fine-tuning                    top-k Mamba blocks (+ final norm)
#   full          Full fine-tuning                                    every Mamba parameter incl. embeddings
#
# All methods adapt only the matrices that every Mamba block owns (in_proj / out_proj) or add small modules; the
# selective-scan parameters (A_log, D, dt_proj, x_proj, conv1d) stay frozen, so the fused scan kernel is unchanged.
# By default the adapters live in the TOP `top_blocks` blocks below the read-out layer: the blocks underneath run
# under no_grad (no activation memory), which is what makes token-mode training feasible on a T4.
#
# Use:  run_experiment(name, mode="token", peft=dict(method="lorma_plus", r=8), ...)   (see the RUN cell)
# ============================================================
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# NormalFloat-4 code book (Dettmers et al., 2023): quantiles of N(0,1) normalised to [-1, 1], with an exact zero.
NF4_TABLE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
             -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
             0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
             0.7229568362236023, 1.0]

PEFT_DEFAULTS = {
    "lorma_plus": dict(r=8, alpha=16, top_blocks=8, targets=("in_proj", "out_proj")),
    # I_pi materialises a (d x d) matrix per layer -> only out_proj (d=512) by default; in_proj has d=3072
    "lorma_pi":   dict(r=8, top_blocks=8, targets=("out_proj",)),
    "adalora":    dict(init_r=12, target_r=6, alpha=32, top_blocks=8, targets=("in_proj", "out_proj"),
                       t_init=0.1, t_final=0.2, delta_t=10, beta=0.85, orth=0.1),
    "qlora":      dict(r=16, alpha=32, top_blocks=8, targets=("in_proj", "out_proj"),
                       quant_all_blocks=True, blocksize=64),
    "adapter":    dict(bottleneck=64, top_blocks=8),
    "prompt":     dict(n_tokens=20),
    "partial":    dict(k=4),
    "full":       dict(),
}


class PEFTHandle:
    """Everything train_model needs to know about the chosen method."""

    def __init__(self, method, spec):
        self.method, self.spec = method, spec
        self.params = []              # NEW trainable parameters created by the method (they get cfg.lr_peft)
        self.mods = []                # injected modules
        self.allocator = None         # AdaLoRA rank allocator

    def param_ids(self):
        return {id(p) for p in self.params}

    def setup(self, total_steps):
        if self.allocator is not None:
            self.allocator.setup(total_steps)

    def extra_loss(self):
        return self.allocator.orth_loss() if self.allocator is not None else 0.0

    def after_backward(self, step):                 # gradients are un-scaled here, before clipping
        if self.allocator is not None:
            self.allocator.update_importance()

    def after_step(self, step):                     # optimizer step done
        if self.allocator is not None:
            self.allocator.after_step(step)

    def describe(self):
        n = sum(p.numel() for p in self.params)
        extra = ""
        if self.allocator is not None:
            r = self.allocator.effective_ranks()
            extra = f" | AdaLoRA ranks now: min {min(r)} / mean {sum(r)/len(r):.1f} / max {max(r)}"
        return f"PEFT '{self.method}': {n/1e6:.3f}M new parameters in {len(self.mods)} modules{extra}"


# ------------------------------------------------------------------ LoRMA
class LoRMALinear(nn.Module):
    """Low-Rank MULTIPLICATIVE Adaptation:  h = I(BA) W0 x     (W0: frozen [d x k],  B: [d x r],  A: [r x d])

    'plus' (LoRMA+): I+(BA) = alpha/r * BA + I_d.   B = 0, A random  ->  identity at init.
                     Evaluated in the cheap order  y + alpha/r * B (A y)  with y = W0 x  (no d x d matrix is built).
    'pi'   (LoRMA-pi): I_pi(BA) rotates row i of BA cyclically by i, which lifts a rank-r matrix to (almost) full rank.
                     Init B[:,0]=1, rest random; A[0,0]=1, rest 0  ->  I_pi(BA) = I_d.   (no alpha in this variant)"""

    def __init__(self, base, r=8, alpha=16, variant="plus"):
        super().__init__()
        assert base.bias is None, "Mamba in_proj/out_proj have no bias"
        d = base.out_features
        self.weight = base.weight                                   # frozen, shares storage with the original layer
        self.variant, self.r, self.scale = variant, r, alpha / r
        self.A, self.B = nn.Parameter(torch.empty(r, d)), nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            self.A.data[0, 0] = 1.0
            nn.init.normal_(self.B, std=0.02)
            self.B.data[:, 0] = 1.0
            idx = (torch.arange(d)[None, :] - torch.arange(d)[:, None]) % d     # M_rot[i, j] = M[i, (j - i) mod d]
            self.register_buffer("rot_idx", idx, persistent=False)

    def forward(self, x):
        y = F.linear(x, self.weight)                                # W0 x
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = torch.gather(self.B @ self.A, 1, self.rot_idx)          # I_pi(BA)  [d x d]
        return F.linear(y, M.to(y.dtype))


# ------------------------------------------------------------------ AdaLoRA
class AdaLoRALinear(nn.Module):
    """W0 x + alpha/r_init * P diag(E) Q x   (SVD-style update; E = singular values, pruned by AdaLoRAAllocator)."""

    def __init__(self, base, r=12, alpha=32):
        super().__init__()
        d, k = base.out_features, base.in_features
        self.weight, self.scale = base.weight, alpha / r
        self.P = nn.Parameter(torch.randn(d, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, k) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))

    def forward(self, x):
        y = F.linear(x, self.weight)
        return y + self.scale * F.linear(F.linear(x, self.Q) * self.E, self.P)


class AdaLoRAAllocator:
    """Importance-based rank allocation (Zhang et al., 2023, Sec. 3.2-3.3).
       sensitivity  I(w) = |w * dL/dw|, smoothed  I_bar <- b I_bar + (1-b) I,  U <- b U + (1-b)|I - I_bar|,  s(w) = I_bar * U
       triplet i    S_i = s(E_i) + mean_j s(P_ji) + mean_j s(Q_ij)
       every delta_t steps the globally lowest-scoring triplets are zeroed so that only b(t) remain, with the cubic
       schedule  b(t) = b_T + (b_0 - b_T) (1 - (t-t_i)/(T-t_i-t_f))^3.  In the last t_f steps the pattern is frozen.
       Orthogonality regulariser  orth * mean_modules( ||P^T P - I||_F + ||Q Q^T - I||_F )."""

    def __init__(self, mods, target_r, t_init, t_final, delta_t, beta, orth):
        self.mods, self.delta_t, self.beta, self.orth = mods, delta_t, beta, orth
        self.t_init_frac, self.t_final_frac = t_init, t_final
        self.b0 = sum(m.E.numel() for m in mods)
        self.bT = int(target_r * len(mods))
        self.ipt, self.unc, self.keep, self.total = {}, {}, None, None

    def setup(self, total_steps):
        self.total = total_steps
        self.ti, self.tf = int(self.t_init_frac * total_steps), int(self.t_final_frac * total_steps)
        self.keep = None

    def orth_loss(self):
        loss = 0.0
        for m in self.mods:
            I = torch.eye(m.P.size(1), device=m.P.device)
            loss = loss + torch.norm(m.P.t() @ m.P - I) + torch.norm(m.Q @ m.Q.t() - I)
        return self.orth * loss / len(self.mods)

    @torch.no_grad()
    def update_importance(self):
        b = self.beta
        for m in self.mods:
            for p in (m.P, m.E, m.Q):
                if p.grad is None:
                    continue
                s = torch.nan_to_num((p * p.grad).abs(), nan=0.0, posinf=0.0, neginf=0.0)
                k = id(p)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(p), torch.zeros_like(p)
                self.ipt[k].mul_(b).add_(s, alpha=1 - b)
                self.unc[k].mul_(b).add_((s - self.ipt[k]).abs(), alpha=1 - b)

    @torch.no_grad()
    def _scores(self):
        out = []
        for m in self.mods:
            sE = self.ipt.get(id(m.E), torch.zeros_like(m.E)) * self.unc.get(id(m.E), torch.zeros_like(m.E))
            sP = self.ipt.get(id(m.P), torch.zeros_like(m.P)) * self.unc.get(id(m.P), torch.zeros_like(m.P))
            sQ = self.ipt.get(id(m.Q), torch.zeros_like(m.Q)) * self.unc.get(id(m.Q), torch.zeros_like(m.Q))
            out.append(sE + sP.mean(0) + sQ.mean(1))                  # [r] per module
        return out

    @torch.no_grad()
    def _prune_to(self, budget):
        scores = self._scores()
        allv = torch.cat(scores)
        if budget >= allv.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        thr = torch.topk(allv, budget).values[-1]
        keep = [s >= thr for s in scores]
        for m, kp in zip(self.mods, keep):
            m.E.data.mul_(kp.to(m.E.dtype))                          # pruned singular values -> 0 (can be revived by gradient)
        return keep

    def budget_at(self, step):
        mul = 1 - (step - self.ti) / max(1, self.total - self.tf - self.ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    @torch.no_grad()
    def after_step(self, step):
        if self.total is None or step <= self.ti:
            return                                                    # warm-up at full rank
        if step > self.total - self.tf:                              # final phase: fixed rank pattern
            if self.keep is None:
                self.keep = self._prune_to(self.bT)
            for m, kp in zip(self.mods, self.keep):
                m.E.data.mul_(kp.to(m.E.dtype))
        elif step % self.delta_t == 0:
            self._prune_to(self.budget_at(step))

    def effective_ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]


# ------------------------------------------------------------------ QLoRA (NF4 base + LoRA)
class NF4Linear(nn.Module):
    """Frozen weight stored as 4-bit NormalFloat codes (two per byte) + one fp16 absmax per `blocksize` weights,
       de-quantised on the fly in the forward pass; optional LoRA update  W x + alpha/r * B A x  (A, B in 16/32 bit).
       Simplifications w.r.t. the QLoRA paper: no double quantisation of the absmax constants, no paged optimiser."""

    def __init__(self, base, r=0, alpha=16, blocksize=64):
        super().__init__()
        w = base.weight.data.detach().float()
        self.out_features, self.in_features, self.blocksize, self.r = base.out_features, base.in_features, blocksize, r
        flat = w.reshape(-1)
        n = flat.numel()
        assert n % blocksize == 0 and (n // 2) * 2 == n
        blk = flat.view(-1, blocksize)
        absmax = blk.abs().amax(1).clamp_min(1e-8)
        normed = (blk / absmax[:, None]).reshape(-1)
        table = torch.tensor(NF4_TABLE, device=w.device)
        codes = torch.empty(n, dtype=torch.uint8, device=w.device)
        for s in range(0, n, 1 << 20):                                # nearest code-book entry, in slices
            codes[s:s + (1 << 20)] = (normed[s:s + (1 << 20), None] - table[None, :]).abs().argmin(1).to(torch.uint8)
        self.register_buffer("qweight", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("table", table, persistent=False)
        self.n = n
        if r > 0:
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, self.in_features))
            self.B = nn.Parameter(torch.zeros(self.out_features, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def dequantize(self, dtype):
        codes = torch.stack([(self.qweight >> 4).long(), (self.qweight & 15).long()], 1).reshape(-1)
        w = self.table[codes].view(-1, self.blocksize) * self.absmax.float()[:, None]
        return w.reshape(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        y = F.linear(x, self.dequantize(x.dtype))
        if self.r > 0:
            y = y + self.scale * F.linear(F.linear(x, self.A), self.B)
        return y


# ------------------------------------------------------------------ bottleneck adapter
class BottleneckAdapter(nn.Module):
    """y + W_up gelu(W_down y), W_up = 0 at init (identity). Mamba has ONE sub-layer per block, so a single adapter per
       block after the mixer (Pfeiffer placement; Houlsby's second adapter has no counterpart here)."""

    def __init__(self, d, bottleneck):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


# ------------------------------------------------------------------ injection
def apply_peft(bb, spec):
    """Inject the method described by spec = dict(method=..., **overrides) into backbone `bb` (a MambaBackbone that
       already froze all pre-trained weights). Returns a PEFTHandle. Call on a deep copy of the Mamba."""
    spec = dict(spec)
    method = spec.pop("method")
    c = {**PEFT_DEFAULTS[method], **spec}
    h = PEFTHandle(method, c)
    n = bb.n_used
    first = 0 if c.get("top_blocks") is None else max(0, n - c["top_blocks"])

    if method == "partial":
        bb.unfreeze_blocks(max(0, n - c["k"]))
    elif method == "full":
        bb.unfreeze_blocks(0)
        bb.train_embed = True
        bb.mamba.token_embedding.weight.requires_grad = True
    elif method == "prompt":
        emb = bb.mamba.token_embedding.weight
        ids = torch.randint(5, emb.size(0), (c["n_tokens"],))        # init from random vocabulary rows (Lester et al.)
        bb.soft_prompt = nn.Parameter(emb.data[ids].clone().float())
        bb.first_grad_block = 0                                      # the prompt sits below every block
        h.params.append(bb.soft_prompt)
    elif method == "adapter":
        for i in range(first, n):
            ad = BottleneckAdapter(bb.d_model, c["bottleneck"])
            bb.mamba.layers[i].adapter = ad
            h.mods.append(ad)
            h.params += list(ad.parameters())
        bb.first_grad_block = min(bb.first_grad_block, first)
    else:                                                            # methods that replace in_proj / out_proj
        quant_from = 0 if (method == "qlora" and c["quant_all_blocks"]) else first
        for i in range(quant_from, n):
            mixer = bb.mamba.layers[i].mixer
            for name in c["targets"]:
                base = getattr(mixer, name)
                adapted = i >= first
                if method in ("lorma_plus", "lorma_pi"):
                    new = LoRMALinear(base, c["r"], c.get("alpha", 16), "plus" if method == "lorma_plus" else "pi")
                    h.params += [new.A, new.B]
                elif method == "adalora":
                    new = AdaLoRALinear(base, c["init_r"], c["alpha"])
                    h.params += [new.P, new.Q, new.E]
                elif method == "qlora":
                    new = NF4Linear(base, c["r"] if adapted else 0, c["alpha"], c["blocksize"])
                    if adapted:
                        h.params += [new.A, new.B]
                else:
                    raise ValueError(method)
                setattr(mixer, name, new)
                if method != "qlora" or adapted:
                    h.mods.append(new)
        bb.first_grad_block = min(bb.first_grad_block, first)
        if method == "adalora":
            h.allocator = AdaLoRAAllocator(h.mods, c["target_r"], c["t_init"], c["t_final"],
                                           c["delta_t"], c["beta"], c["orth"])
    return h


# ------------------------------------------------------------------ size / cost accounting
def mamba_prefix_footprint(mamba, n_used):
    """(#params, bytes) of the embedding + the first n_used blocks (+ their PEFT modules and 4-bit buffers)."""
    params = nbytes = 0
    for name, p in mamba.named_parameters():
        mt = re.match(r"layers\.(\d+)\.", name)
        if mt and int(mt.group(1)) >= n_used:
            continue
        params += p.numel(); nbytes += p.numel() * p.element_size()
    for name, m in mamba.named_modules():
        mt = re.match(r"layers\.(\d+)\.", name)
        if isinstance(m, NF4Linear) and not (mt and int(mt.group(1)) >= n_used):
            params += m.n; nbytes += m.qweight.numel() + m.absmax.numel() * 2
    return params, nbytes


def model_footprint(model, mode, n_used_cached=None):
    """Deployable pipeline = Mamba prefix (embedding + blocks that are used) + everything trained on top of it."""
    if mode == "token":
        bb = model.backbone
        p0, b0 = mamba_prefix_footprint(bb.mamba, bb.n_used)
        head = [(n, p) for n, p in model.named_parameters() if not n.startswith("backbone.")]
        extra = [bb.soft_prompt] if bb.soft_prompt is not None else []
    else:
        p0, b0 = mamba_prefix_footprint(MAMBA, n_used_cached)
        head, extra = list(model.named_parameters()), []
    pn = sum(p.numel() for _, p in head) + sum(p.numel() for p in extra)
    bn = sum(p.numel() * p.element_size() for _, p in head) + sum(p.numel() * p.element_size() for p in extra)
    return (p0 + pn) / 1e6, (b0 + bn) / 1e6

In [ ]:
# ============================================================
# 3a. FAST-PATH SELF-TEST (run once, after the data is loaded and the heads above are defined)
#
# Compares three ways of computing the SAME Mamba hidden states on real training chunks:
#   reference : Phase-1 maths - pure-PyTorch chunked scan, fp32            (always available)
#   kernel    : fused mamba_ssm selective scan, fp32                       (if installed + GPU)
#   kernel+AMP: fused scan with fp16/bf16 autocast for projections/conv   (if GPU)
# It prints time, tokens/s and the deviation from the reference, and switches the kernel and/or
# half precision OFF by itself when they disagree with the reference (so a bad build can never
# silently corrupt the cached features).
# ============================================================
def pick_amp_dtype():
    if device.type != "cuda":
        return None
    # bf16 only on Ampere+ (a T4 would emulate it slowly) -> fp16 on T4/V100
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


AMP_DTYPE = pick_amp_dtype()
print("AMP dtype:", AMP_DTYPE)


@torch.no_grad()
def check_fast_path(n=8, T=512):
    global AMP_DTYPE
    docs = [d for d in enc_train if len(d) >= T][:n]
    assert docs, "no training document has >= 512 tokens"
    ids = torch.as_tensor(np.stack([d[:T] for d in docs]), dtype=torch.long, device=device)
    bb = MambaBackbone(MAMBA.to(device), -1, 0).to(device).eval()
    ntok = ids.numel()
    sync = (lambda: torch.cuda.synchronize()) if device.type == "cuda" else (lambda: None)

    def timed(fn):
        fn(); sync()                                          # warm-up: CUDA context, cuDNN autotune, kernel load
        t = time.time(); out = fn().float(); sync()
        return out, time.time() - t

    def run(scan_kernel, amp):
        FAST["scan_kernel"] = scan_kernel
        def f():
            with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=amp and AMP_DTYPE is not None):
                return bb(ids)
        return timed(f)

    cos = lambda a, b: F.cosine_similarity(a.reshape(-1, a.size(-1)), b.reshape(-1, b.size(-1)), dim=-1).min().item()
    kernel_ok = bool(HAS_SSM_KERNEL and device.type == "cuda")

    ref, t_ref = run(False, False)
    print(f"reference  (PyTorch scan, fp32)      : {t_ref*1000:8.0f} ms | {ntok/t_ref:9,.0f} tok/s")

    use_kernel = False
    if kernel_ok:
        k32, t_k32 = run(True, False)
        c = cos(ref, k32)
        print(f"kernel     (fused scan, fp32)        : {t_k32*1000:8.0f} ms | {ntok/t_k32:9,.0f} tok/s | "
              f"max|diff| {(ref-k32).abs().max():.2e} | min cos {c:.6f} | speed-up x{t_ref/t_k32:.1f}")
        use_kernel = c > 0.999 and bool(torch.isfinite(k32).all())
        if not use_kernel:
            print("WARNING: fused kernel output differs from the Phase-1 reference -> kernel switched OFF")
    else:
        print("fused kernel                         : not used (" + ("no GPU" if device.type != "cuda" else "mamba_ssm missing") + ")")

    if AMP_DTYPE is not None:
        h16, t16 = run(use_kernel, True)
        c16 = cos(ref, h16)
        print(f"{'kernel' if use_kernel else 'PyTorch scan'}+AMP ({str(AMP_DTYPE).split('.')[-1]})".ljust(37) +
              f": {t16*1000:8.0f} ms | {ntok/t16:9,.0f} tok/s | max|diff| {(ref-h16).abs().max():.2e} | "
              f"min cos {c16:.6f} | speed-up x{t_ref/t16:.1f}")
        finite = bool(torch.isfinite(h16).all())
        print("NaN/Inf in AMP output:", not finite)
        if c16 < 0.99 or not finite:
            print("WARNING: mixed precision is hurting the features -> AMP switched OFF (AMP_DTYPE=None)")
            AMP_DTYPE = None

    FAST["scan_kernel"] = use_kernel
    print(f"\n=> scan kernel: {'ON' if use_kernel else 'OFF'} | AMP: {AMP_DTYPE}")
    del bb, ref
    if device.type == "cuda":
        torch.cuda.empty_cache()


check_fast_path()

AMP dtype: torch.bfloat16
reference  (PyTorch scan, fp32)      :      399 ms |    10,265 tok/s
fused kernel                         : not used (mamba_ssm missing)
PyTorch scan+AMP (bfloat16)          :      376 ms |    10,904 tok/s | max|diff| 2.97e-02 | min cos 0.999971 | speed-up x1.1
NaN/Inf in AMP output: False

=> scan kernel: OFF | AMP: torch.bfloat16


In [ ]:
import torch, sys
print(torch.__version__, torch.version.cuda, sys.version.split()[0])

2.11.0+cu130 13.0 3.13.16


In [ ]:
# ============================================================
# 3. DATASETS, COLLATE, MAMBA FEATURE EXTRACTION
# ============================================================
class ChunkedDocs(Dataset):
    """Each document -> list of <=512-token chunks (after the chosen long-document strategy)."""

    def __init__(self, enc_docs, y, cfg):
        self.items = [select_chunks(chunk_ids(d, cfg.chunk_len, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy)
                      for d in enc_docs]
        self.y = y

    def __len__(self): return len(self.items)

    def __getitem__(self, i): return i, self.items[i], (None if self.y is None else self.y[i])


class FeatDocs(Dataset):
    def __init__(self, feats, y):
        self.feats, self.y = feats, y

    def __len__(self): return len(self.feats)

    def __getitem__(self, i): return i, self.feats[i], (None if self.y is None else self.y[i])


def _stack_y(batch):
    return None if batch[0][2] is None else torch.from_numpy(np.stack([b[2] for b in batch]))


def collate_tokens(batch):
    B = len(batch)
    N = max(len(c) for _, c, _ in batch)
    T = max(len(x) for _, c, _ in batch for x in c)
    ids = torch.full((B, N, T), PAD_ID, dtype=torch.long)
    tm = torch.zeros(B, N, T, dtype=torch.bool)
    cm = torch.zeros(B, N, dtype=torch.bool)
    for b, (_, chunks, _) in enumerate(batch):
        for n, c in enumerate(chunks):
            ids[b, n, :len(c)] = torch.as_tensor(np.asarray(c), dtype=torch.long)
            tm[b, n, :len(c)] = True
            cm[b, n] = True
    return dict(ids=ids, tok_mask=tm, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))


def collate_feats(batch):
    B, N, F_ = len(batch), max(f.size(0) for _, f, _ in batch), batch[0][1].size(1)
    x = torch.zeros(B, N, F_)
    cm = torch.zeros(B, N, dtype=torch.bool)
    for b, (_, f, _) in enumerate(batch):
        x[b, :f.size(0)] = f.float()
        cm[b, :f.size(0)] = True
    return dict(feats=x, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))


@torch.no_grad()
def extract_chunk_features(mamba, ds, tok_budget=None, max_bs=256, use_layer=-1, amp=True, log_every=25):
    """Run the frozen Mamba once over every chunk -> fp16 [N_i, 3*d] per document.
    Batches are built from a TOKEN budget over length-sorted chunks (short chunks -> bigger
    batches), under fp16/bf16 autocast, with the fused Mamba scan kernel when the self-test
    (cell 3a) approved it. Right-padding is safe: scan and conv are causal, so real tokens
    never see the pad positions.
    tok_budget=None -> 32768 tokens/batch with the fused kernel, 8192 with the PyTorch fallback
    (its scan intermediates are far larger); OOM halves the batch automatically either way."""
    if tok_budget is None:
        tok_budget = 32768 if FAST["scan_kernel"] else 8192
    bb = MambaBackbone(mamba, use_layer, 0).to(device).eval()
    flat = [c for chunks in ds.items for c in chunks]                 # doc-major order
    doc_len = [len(chunks) for chunks in ds.items]
    n, d = len(flat), bb.d_model
    lens = np.fromiter((len(c) for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")                           # similar lengths -> little padding
    out = torch.empty(n, 3 * d, dtype=torch.float16)                  # one preallocated buffer
    use_amp = amp and AMP_DTYPE is not None
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens | amp={AMP_DTYPE if use_amp else 'off'} | "
          f"scan kernel={'on' if FAST['scan_kernel'] else 'off'} | budget {tok_budget} tok/batch")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            arr[j, :lens[i]] = flat[i]
        ids = torch.from_numpy(arr).to(device, non_blocking=True)
        ln = torch.from_numpy(lens[sel]).to(device)
        with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=use_amp):
            h = bb(ids)
        h = h.float()
        m = (torch.arange(T, device=device)[None, :] < ln[:, None]).unsqueeze(-1)
        mean = (h * m).sum(1) / ln[:, None]
        mx = h.masked_fill(~m, -1e4).max(1).values
        last = h[torch.arange(len(sel), device=device), ln - 1]
        out[torch.from_numpy(sel)] = torch.cat([mean, mx, last], -1).half().cpu()

    def safe_run(sel):                                                # halve the batch on CUDA OOM
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache()
            mid = len(sel) // 2
            safe_run(sel[:mid]); safe_run(sel[mid:])

    t0, done_tok, nb, s = time.time(), 0, 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        sel = order[s:e]
        safe_run(sel)
        done_tok += int(lens[sel].sum()); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            dt = time.time() - t0
            eta = dt * (int(lens.sum()) - done_tok) / max(done_tok, 1)
            print(f"  features {s:,}/{n:,} chunks | {done_tok/dt:,.0f} tok/s | elapsed {dt/60:.1f} min | ETA {eta/60:.1f} min")

    feats, p = [], 0
    for k in doc_len:                                                 # per-doc views of the one buffer
        feats.append(out[p:p + k]); p += k
    return feats


def get_features(split, enc_docs, cfg):
    # 'mamba' + checkpoint signature in the name: can never be confused with cached features of another backbone
    sig = f"mamba{CKPT_SIG}_{split}_L{cfg.chunk_len}_S{cfg.chunk_stride}_K{cfg.max_chunks}_{cfg.chunk_strategy}_layer{cfg.use_layer}"
    name = f"feat_{sig}.pt"
    drive_path = os.path.join(p5_dir, name)
    local_dir = "/content/feat_cache"; os.makedirs(local_dir, exist_ok=True)   # local disk is much faster than Drive
    local_path = os.path.join(local_dir, name)
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
    if os.path.exists(local_path):
        return torch.load(local_path, weights_only=False)
    print(f"extracting Mamba features for '{split}' ...")
    ds = ChunkedDocs(enc_docs, None, cfg)
    feats = extract_chunk_features(MAMBA.to(device), ds, use_layer=cfg.use_layer)
    torch.save(feats, local_path)
    shutil.copy(local_path, drive_path)                                        # persist to Drive once
    return feats

In [ ]:
# ============================================================
# 4. LOSSES, TRAINING, EVALUATION
# ============================================================
class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021): down-weights the many easy negatives of a multi-label problem."""

    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        xs_neg = (1 - p + self.clip).clamp(max=1) if self.clip > 0 else 1 - p
        los = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        pt = p * y + xs_neg * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(los * (1 - pt) ** gamma).sum() / logits.size(0)


def build_loss(cfg, y_tr):
    if cfg.loss == "bce":
        return nn.BCEWithLogitsLoss()
    if cfg.loss == "bce_pw":      # rare statutes get a larger positive weight: ((N-n_c)/n_c)^power, clipped
        pos = y_tr.sum(0)
        pw = np.clip(((len(y_tr) - pos) / np.maximum(pos, 1)) ** cfg.pw_power, 1.0, cfg.pw_max)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32))
    if cfg.loss == "asl":
        return AsymmetricLoss()
    raise ValueError(cfg.loss)


def forward_batch(model, batch, mode):
    cm = batch["chunk_mask"].to(device)
    if mode == "token":
        return model(batch["ids"].to(device), batch["tok_mask"].to(device), cm)
    return model(batch["feats"].to(device), cm)


def resolve_amp(cfg, mode):
    """cfg.amp: None -> on for 'token' mode (the Mamba forward runs every step), off for 'cached' (tiny head);
    True/False forces it. Always off when the self-test disabled half precision or there is no GPU."""
    want = (mode == "token") if getattr(cfg, "amp", None) is None else bool(cfg.amp)
    return want and AMP_DTYPE is not None and device.type == "cuda"


def amp_context(enabled):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=enabled)


@torch.no_grad()
def predict_logits(model, loader, mode, amp=False):
    model.eval()
    out = []
    for b in loader:
        with amp_context(amp):
            logits = forward_batch(model, b, mode)
        out.append(logits.float().cpu().numpy())
    return np.concatenate(out)


def predict_probs(model, loader, mode, amp=False):
    return 1.0 / (1.0 + np.exp(-np.clip(predict_logits(model, loader, mode, amp), -30, 30)))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_model(model, mode, train_loader, dev_loader, y_dev, cfg, y_tr, tag):
    loss_fn = build_loss(cfg, y_tr).to(device)
    peft = getattr(model, "peft", None)
    peft_ids = peft.param_ids() if peft is not None else set()
    new_p = [p for n, p in model.named_parameters() if p.requires_grad and id(p) in peft_ids]       # new PEFT params
    bb = [p for n, p in model.named_parameters()
          if p.requires_grad and n.startswith("backbone.") and id(p) not in peft_ids]               # pre-trained weights being tuned
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]  # BiRNN heads
    groups = ([dict(params=rest, lr=cfg.lr)] + ([dict(params=new_p, lr=cfg.lr_peft)] if new_p else [])
              + ([dict(params=bb, lr=cfg.lr_backbone)] if bb else []))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    total = cfg.epochs * math.ceil(len(train_loader) / cfg.accum)
    if peft is not None:
        peft.setup(total)
        print(f"[{tag}] {peft.describe()}")
    kd = getattr(cfg, "kd", None)                       # knowledge distillation: dict(teachers=[names], T=2.0, lam=0.5)
    if kd:
        t_logits = torch.from_numpy(cfg.kd_teacher_logits)          # [N_train, L], same order as the training docs
        print(f"[{tag}] distilling from {kd['teachers']} | T={kd['T']} lambda={kd['lam']}")
    gstep = 0
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))

    use_amp = resolve_amp(cfg, mode)
    # fp16 needs loss scaling; bf16 (Ampere+) does not. Disabled scaler = plain backward/step.
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and AMP_DTYPE == torch.float16)
    print(f"[{tag}] autocast: {AMP_DTYPE if use_amp else 'off'} | grad scaler: {scaler.is_enabled()}")

    best, best_state, bad = -1.0, None, 0
    for ep in range(cfg.epochs):
        model.train()
        t0, run, step = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            with amp_context(use_amp):
                logits = forward_batch(model, b, mode)
            logits, y = logits.float(), b["y"].to(device)               # losses always in fp32
            loss = loss_fn(logits, y)
            if kd:                                                      # (1-lam) * hard + lam * T^2 * soft (sigmoid targets)
                T = kd["T"]
                tl = t_logits[b["idx"]].to(device)
                soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(tl / T)) * (T * T)
                loss = (1 - kd["lam"]) * loss + kd["lam"] * soft
            if peft is not None:
                loss = loss + peft.extra_loss()                         # AdaLoRA orthogonality regulariser
            scaler.scale(loss / cfg.accum).backward()
            run += loss.item()
            if (i + 1) % cfg.accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(opt)
                if peft is not None:
                    peft.after_backward(gstep)                          # AdaLoRA: update importance scores
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                gstep += 1
                if peft is not None:
                    peft.after_step(gstep)                              # AdaLoRA: prune ranks to the budget schedule
            if ep == 0 and i + 1 == 20:                                 # cost estimate after 20 mini-batches
                sp = (time.time() - t0) / 20
                print(f"[{tag}] {sp:.2f} s/batch -> ~{sp*len(train_loader)/60:.1f} min per epoch")
        probs = predict_probs(model, dev_loader, mode, use_amp)
        thr, macro = tune_global_threshold(probs, y_dev)
        m = compute_metrics(y_dev, predict_labels(probs, thr))
        print(f"[{tag}] ep {ep+1:02d}  loss {run/len(train_loader):.4f}  dev macro-F1 {macro:.4f}  "
              f"micro-F1 {m['micro_f1']:.4f}  subset {m['subset_acc']:.4f}  thr {thr:.3f}  ({time.time()-t0:.0f}s)")
        if macro > best:
            best, bad = macro, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    model.load_state_dict(best_state, strict=False)           # frozen Mamba weights are untouched
    return best_state


BASE = dict(
    # long-document representation
    chunk_len=512, chunk_stride=512, max_chunks=16, chunk_strategy="head_tail",
    # Mamba features
    use_layer=-1, unfreeze_top_k=0, mamba_chunk_bs=8,
    # bidirectional heads
    tok_rnn="gru", tok_hid=128, chunk_rnn="gru", chunk_hid=256, chunk_layers=1, pool="attn", dropout=0.3,
    # loss
    loss="bce_pw", pw_power=0.5, pw_max=30.0,
    # optimisation
    lr=1e-3, lr_backbone=2e-5, lr_peft=5e-4, wd=0.01, epochs=20, patience=5, batch_size=32, accum=1, warmup=0.06, clip=1.0,
    seed=42, min_support=3,
    # mixed precision: None = auto (on for mode='token', off for 'cached'); True/False forces it
    amp=None,
    # adaptation method for mode='token' (see 'PEFT for Mamba'): None | dict(method=..., **overrides); distillation: dict(teachers=[...], T=2.0, lam=0.5)
    peft=None, kd=None,
)

RESULTS = {}


def run_experiment(name, mode="cached", keep_train_logits=False, **over):
    """mode='cached' : Mamba run once -> train BiRNN over chunk features (fast; no Mamba adaptation).
       mode='token'  : Mamba states -> token-level BiRNN -> chunk-level BiRNN; cfg.peft adapts the Mamba itself.
       keep_train_logits=True stores the model's logits on the TRAIN documents (needed if it will be a KD teacher)."""
    cfg = SimpleNamespace(**{**BASE, **over})
    set_seed(cfg.seed)
    if cfg.peft and mode != "token":
        raise ValueError("peft adapts the Mamba itself -> use mode='token'")
    if mode == "cached":
        ftr, fdv, fte = (get_features(s, e, cfg) for s, e in
                         (("train", enc_train), ("dev", enc_dev), ("test", enc_test)))
        d_tr, d_dv, d_te = FeatDocs(ftr, y_train), FeatDocs(fdv, y_dev), FeatDocs(fte, None)
        coll, model = collate_feats, CachedFeatureClassifier(ftr[0].size(1), L, cfg)
        g = None
    else:
        d_tr, d_dv, d_te = (ChunkedDocs(enc_train, y_train, cfg), ChunkedDocs(enc_dev, y_dev, cfg),
                            ChunkedDocs(enc_test, None, cfg))
        g = copy.deepcopy(MAMBA) if (cfg.peft or cfg.unfreeze_top_k > 0) else MAMBA   # never modify the shared Phase-1 model
        coll, model = collate_tokens, MambaBiRNNClassifier(g, L, cfg)
    if cfg.kd:                                                       # teacher logits on the training docs (averaged)
        miss = [t for t in cfg.kd["teachers"] if "train_logits" not in RESULTS.get(t, {})]
        assert not miss, f"teacher(s) {miss}: run them first with keep_train_logits=True"
        cfg.kd_teacher_logits = np.mean([RESULTS[t]["train_logits"] for t in cfg.kd["teachers"]], axis=0).astype(np.float32)
    model.to(device)
    mk = lambda d, sh: DataLoader(d, batch_size=cfg.batch_size if sh else max(cfg.batch_size, 8), shuffle=sh,
                                  collate_fn=coll, num_workers=0, drop_last=False)
    tr, dv, te, tr_eval = mk(d_tr, True), mk(d_dv, False), mk(d_te, False), mk(d_tr, False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_used = model.backbone.n_used if mode == "token" else (len(MAMBA.layers) if cfg.use_layer == -1 else cfg.use_layer + 1)
    params_M, size_MB = model_footprint(model, mode, n_used)
    print(f"\n=== {name} | mode={mode} | trainable params {n_train/1e6:.3f}M | deployed pipeline {params_M:.1f}M params "
          f"/ {size_MB:.0f} MB ({n_used} Mamba blocks) ===")

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_model(model, mode, tr, dv, y_dev, cfg, y_train, name)
    train_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0

    amp_on = resolve_amp(cfg, mode)
    t0 = time.time()
    lg_dev = predict_logits(model, dv, mode, amp_on)
    infer_s = time.time() - t0                                        # dev-set inference wall-clock (incl. Mamba forward in token mode)
    p_dev = 1.0 / (1.0 + np.exp(-np.clip(lg_dev, -30, 30)))
    p_test = predict_probs(model, te, mode, amp_on)
    g_thr, _ = tune_global_threshold(p_dev, y_dev)
    pc_thr = tune_per_class_threshold(p_dev, y_dev, g_thr, min_support=cfg.min_support)
    res = dict(cfg={k: v for k, v in vars(cfg).items() if k != "kd_teacher_logits"}, mode=mode, p_dev=p_dev, p_test=p_test,
               g_thr=g_thr, pc_thr=pc_thr, trainable_params=n_train, deployed_params_M=params_M, deployed_size_MB=size_MB,
               train_seconds=train_s, dev_infer_seconds=infer_s, peak_gpu_MB=peak_mb,
               m_fixed=compute_metrics(y_dev, predict_labels(p_dev, 0.5)),
               m_global=compute_metrics(y_dev, predict_labels(p_dev, g_thr)),
               m_perclass=compute_metrics(y_dev, predict_labels(p_dev, pc_thr)),
               honest=honest_threshold_eval(p_dev, y_dev))
    if getattr(model, "peft", None) is not None and model.peft.allocator is not None:
        res["adalora_ranks"] = model.peft.allocator.effective_ranks()
    if keep_train_logits:
        res["train_logits"] = predict_logits(model, tr_eval, mode, amp_on)
    RESULTS[name] = res
    pickle.dump({k: v for k, v in res.items()}, open(os.path.join(p5_dir, f"result_{name}.pkl"), "wb"))
    print(f"[{name}] dev macro-F1  thr=0.5: {res['m_fixed']['macro_f1']:.4f} | global thr {g_thr:.3f}: "
          f"{res['m_global']['macro_f1']:.4f} | per-class: {res['m_perclass']['macro_f1']:.4f} (optimistic)")
    print(f"[{name}] honest split-half macro-F1 (tune on half, score on other half): {res['honest']}")
    print(f"[{name}] cost: train {train_s/60:.1f} min | dev inference {infer_s:.1f}s | peak GPU {peak_mb:.0f} MB")
    del model, g
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res


def write_submission(name, path, per_class=False):
    r = RESULTS[name]
    thr = r["pc_thr"] if per_class else r["g_thr"]
    pred = predict_labels(r["p_test"], thr)
    out = [dict(id=rec["id"], pred=sorted(int(idx2lab[j]) for j in np.flatnonzero(p)))
           for rec, p in zip(test_recs, pred)]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one prediction per test id"
    assert all(len(o["pred"]) >= 1 and len(set(o["pred"])) == len(o["pred"]) for o in out)
    json.dump(out, open(path, "w"), ensure_ascii=False)
    print(f"wrote {len(out)} predictions -> {path}  (avg {np.mean([len(o['pred']) for o in out]):.2f} statutes/doc)")

In [ ]:
# ---- TRIAL-SIZE TRAINING SET (all six experiments train on the same subset) -----------------------------------
SUB_N = 4000                                   # documents to keep; ~157.7 min * SUB_N/42500 per epoch at max_chunks=2
if "enc_train_full" not in globals():          # keep the originals so this cell is safe to re-run
    enc_train_full, y_train_full = enc_train, y_train
    _get_features_orig = get_features

rng = np.random.RandomState(0)
freq_full = y_train_full.sum(0)
rarity = (y_train_full / np.maximum(freq_full, 1)).max(1)          # docs that carry a rare statute score high
order = np.argsort(-rarity, kind="stable")
n_rare = SUB_N // 3                                                 # a third of the budget goes to the rarest docs ...
keep = np.concatenate([order[:n_rare], rng.permutation(order[n_rare:])[:SUB_N - n_rare]])   # ... the rest is random
keep = np.sort(keep)

enc_train = [enc_train_full[i] for i in keep]
y_train = y_train_full[keep]


def get_features(split, enc_docs, cfg):                            # separate cache file, so the full-train features are never mixed in
    return _get_features_orig(f"train_sub{SUB_N}" if split == "train" else split, enc_docs, cfg)


print(f"training on {len(enc_train):,} of {len(enc_train_full):,} documents | "
      f"statutes with >=1 positive: {int((y_train.sum(0) > 0).sum())} of {y_train.shape[1]}")

training on 4,000 of 42,750 documents | statutes with >=1 positive: 100 of 100


In [ ]:
from contextlib import nullcontext

def _forward_reentrant_ckpt(self, ids):                   # ids [M,T] -> [M,T,d]
    m = self.mamba
    lo = min(self.first_grad_block, self.n_used)
    with torch.set_grad_enabled(self.train_embed and torch.is_grad_enabled()):
        h = m.token_embedding(ids)
    P = 0
    if self.soft_prompt is not None:
        P = self.soft_prompt.size(0)
        h = torch.cat([self.soft_prompt.unsqueeze(0).expand(h.size(0), -1, -1).to(h.dtype), h], dim=1)
    with torch.no_grad() if lo > 0 else nullcontext():     # frozen lower blocks: no graph, no memory
        for blk in m.layers[:lo]:
            h = blk(h)
    for blk in m.layers[lo:self.n_used]:                   # blocks that need gradients
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            if not h.requires_grad:                        # reentrant checkpoint needs a grad-requiring input
                h = h.detach().requires_grad_(True)
            h = grad_checkpoint(blk, h, use_reentrant=True)
        else:
            h = blk(h)
    h = m.norm_final(h) if self.use_layer == -1 else rms_normalize(h)
    return h[:, P:] if P else h

MambaBackbone.forward = _forward_reentrant_ckpt
print("backbone: reentrant gradient checkpointing")

backbone: reentrant gradient checkpointing


In [ ]:
# ============================================================
# 5. RUN  -  the six Phase-5 experiments for the Mamba architecture
#
#   baseline  frozen Mamba + BiGRU on cached features                (zero adaptation of the Mamba; NOT one of the six)
#   E1  LoRMA+            multiplicative low-rank adaptation of in_proj/out_proj          [compulsory technique]
#   E2  AdaLoRA           SVD-style low-rank update with importance-based rank pruning
#   E3  QLoRA             frozen 4-bit NF4 Mamba + LoRA
#   E4  Adapters          bottleneck adapter after every mixer (Houlsby/Pfeiffer-style)
#   E5  Partial FT        fine-tune the top-k Mamba blocks (layer-wise fine-tuning)
#   E6  Knowledge dist.   teacher = ensemble of E1-E5 (20-block Mamba + adapters), student = 8-block Mamba + small BiGRU
#
# That is 5 different fine-tuning / PEFT techniques + 1 distillation experiment (brief, sec. 11.2).
# Optional extras below: LoRMA-pi, prompt tuning, full fine-tuning, student without distillation.
#
# !! Every number below (epochs, ranks, learning rates, chunks, temperature, ...) is only a STARTING POINT so that the
# !! notebook runs end-to-end. Tuning them and interpreting the results is your job (and the course rules say so).
# !! Run ONE experiment per cell first and read the "s/batch -> ~N min per epoch" line before queueing the rest.
# ============================================================
TOK = dict(mode="token", batch_size=4, accum=4, max_chunks=2, epochs=4, patience=2,
           lr=1e-3, lr_peft=5e-4, lr_backbone=2e-5, mamba_chunk_bs=8)

# ---- baseline: frozen Mamba, cached features (fast; shows what the PEFT methods have to beat) -----------------
# r_base = run_experiment("cached_bigru", mode="cached")

# ---- E1  LoRMA+  (compulsory) -------------------------------------------------------------------------------------
r1 = run_experiment("e1_lorma_plus", keep_train_logits=True,
                    peft=dict(method="lorma_plus", r=8, alpha=16, top_blocks=8), **TOK)

# ---- E2  AdaLoRA --------------------------------------------------------------------------------------------------
r2 = run_experiment("e2_adalora", keep_train_logits=True,
                    peft=dict(method="adalora", init_r=12, target_r=6, top_blocks=8), **TOK)

# ---- E3  QLoRA  (4-bit NF4 base for all used blocks, LoRA in the top blocks) ----------------------------------------
r3 = run_experiment("e3_qlora", keep_train_logits=True,
                    peft=dict(method="qlora", r=16, alpha=32, top_blocks=8), **TOK)

# ---- E4  Bottleneck adapters ----------------------------------------------------------------------------------------
r4 = run_experiment("e4_adapter", keep_train_logits=True,
                    peft=dict(method="adapter", bottleneck=64, top_blocks=8), **TOK)

# ---- E5  Partial / layer-wise fine-tuning (pre-trained weights of the top 4 blocks are updated) -------------------------
r5 = run_experiment("e5_partial_ft", keep_train_logits=True,
                    peft=dict(method="partial", k=4), **TOK)

# ---- E6  Knowledge distillation -------------------------------------------------------------------------------------
# Teacher : ensemble (mean logit) of E1-E5 = 20 Mamba blocks + adapters + hierarchical BiRNN head.
# Student : the FIRST 8 Mamba blocks (use_layer=7) frozen + a smaller BiGRU head (chunk_hid=128); features cached.
#           -> 8/20 of the Mamba compute and memory at inference. Loss = (1-lam)*weighted-BCE + lam*T^2*BCE(student/T, sigmoid(teacher/T)).
r6 = run_experiment("e6_kd_student", mode="cached", use_layer=7, chunk_hid=128,
                    kd=dict(teachers=["e1_lorma_plus", "e2_adalora", "e3_qlora", "e4_adapter", "e5_partial_ft"], T=2.0, lam=0.5))

# ---- optional extras --------------------------------------------------------------------------------------------------
# r6b = run_experiment("e6b_student_no_kd", mode="cached", use_layer=7, chunk_hid=128)        # same student, no teacher (isolates the KD effect)
# r7  = run_experiment("x_lorma_pi", keep_train_logits=True, peft=dict(method="lorma_pi", r=8, top_blocks=8), **TOK)
# r8  = run_experiment("x_prompt", peft=dict(method="prompt", n_tokens=20), **{**TOK, "lr_peft": 5e-2})     # backward through ALL blocks: slow
# r9  = run_experiment("x_full_ft", peft=dict(method="full"), **{**TOK, "batch_size": 2, "accum": 8})        # all params + embeddings: slow, needs memory
# Other levers from the brief on the same code: loss=("bce"|"bce_pw"|"asl"), chunk_strategy, max_chunks, pool="label_attn",
# chunk_rnn=("gru"|"lstm"|"transformer"), use_layer, and the thresholds (global / per class) in the summary below.

# ---- summary table (accuracy + cost) ------------------------------------------------------------------------------------------
print(f"\n{'experiment':20s} {'macro-F1':>8s} {'micro-F1':>8s} {'subset':>7s} {'honest(g)':>9s} {'train M':>8s} "
      f"{'deploy M':>8s} {'MB':>6s} {'train min':>9s} {'dev inf s':>9s} {'peak MB':>8s}")
for n, r in RESULTS.items():
    m = r["m_global"]
    print(f"{n:20s} {m['macro_f1']:8.4f} {m['micro_f1']:8.4f} {m['subset_acc']:7.4f} {r['honest']['global']:9.4f} "
          f"{r['trainable_params']/1e6:8.3f} {r['deployed_params_M']:8.1f} {r['deployed_size_MB']:6.0f} "
          f"{r['train_seconds']/60:9.1f} {r['dev_infer_seconds']:9.1f} {r['peak_gpu_MB']:8.0f}")
if "e2_adalora" in RESULTS:
    print("AdaLoRA final ranks per adapted matrix:", RESULTS["e2_adalora"]["adalora_ranks"])

# ---- rare-statute analysis for one run ----------------------------------------------------------------------------------
best_name = max(RESULTS, key=lambda n: RESULTS[n]["honest"]["global"])
r = RESULTS[best_name]
print(f"\nF1 by number of TRAIN docs per statute ({best_name}):")
for bucket, n_cls, f1 in rare_report(y_dev, predict_labels(r["p_dev"], r["g_thr"]), y_train):
    print(f"  train-freq {bucket:>8s}: {n_cls:4d} statutes  mean F1 {f1:.3f}")


# ---- Codabench packages: prediction.json at the ROOT of submission_<name>.zip -------------------------------------------------
import zipfile


def make_submission_zip(name, per_class=None):
    """Writes prediction.json for experiment `name` and zips it (flat, as Codabench expects).
       per_class=None -> use per-class thresholds only if their honest split-half score beats the global threshold."""
    r = RESULTS[name]
    use_pc = (r["honest"]["per_class"] > r["honest"]["global"]) if per_class is None else per_class
    d = os.path.join(p5_dir, f"sub_{name}"); os.makedirs(d, exist_ok=True)
    write_submission(name, os.path.join(d, "prediction.json"), per_class=use_pc)
    zpath = os.path.join(p5_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), arcname="prediction.json")
    print(f"  -> {zpath}")
    return zpath


# One zip per experiment (the brief needs >= 6 distinct, non-identical submissions):
for name in RESULTS:
    make_submission_zip(name)


=== e1_lorma_plus | mode=token | trainable params 2.582M | deployed pipeline 72.1M params / 288 MB (20 Mamba blocks) ===
[e1_lorma_plus] PEFT 'lorma_plus': 0.459M new parameters in 16 modules
[e1_lorma_plus] autocast: torch.bfloat16 | grad scaler: False
[e1_lorma_plus] 0.79 s/batch -> ~13.1 min per epoch
[e1_lorma_plus] ep 01  loss 0.4981  dev macro-F1 0.0730  micro-F1 0.0969  subset 0.0000  thr 0.100  (1591s)
[e1_lorma_plus] ep 02  loss 0.4685  dev macro-F1 0.0748  micro-F1 0.0849  subset 0.0000  thr 0.075  (1592s)
[e1_lorma_plus] ep 03  loss 0.4614  dev macro-F1 0.0785  micro-F1 0.0997  subset 0.0000  thr 0.100  (1591s)
[e1_lorma_plus] ep 04  loss 0.4573  dev macro-F1 0.0785  micro-F1 0.0944  subset 0.0000  thr 0.100  (1592s)
[e1_lorma_plus] dev macro-F1  thr=0.5: 0.0099 | global thr 0.100: 0.0785 | per-class: 0.0850 (optimistic)
[e1_lorma_plus] honest split-half macro-F1 (tune on half, score on other half): {'fixed_0.5': 0.009862376842647791, 'global': 0.07846357673406601, 'per_cla

In [ ]:
import glob, os, torch
bad = []
for d in ("/content/feat_cache", p5_dir):
    for f in glob.glob(os.path.join(d, "feat_*.pt")):
        try:
            torch.load(f, weights_only=False)
        except Exception as e:
            bad.append(f); os.remove(f)
            print("deleted corrupt:", f)
print("corrupt files removed:", len(bad))

corrupt files removed: 0
